In [0]:
-- ==============================================================================
-- GOLD LAYER DATA QUALITY CHECKS
-- ==============================================================================

-- ------------------------------------------------------------------------------
-- CHECK 1: Volumetric & Revenue Reconciliation (Silver vs. Gold)
-- Objective: Ensure no records or revenue were lost/duplicated during ETL
-- ------------------------------------------------------------------------------

SELECT 
    'Silver (cleansed_sales)' AS layer, 
    COUNT(*) AS total_records, 
    SUM(revenue) AS total_revenue
FROM ecommerce_silver.cleansed_sales
UNION ALL
SELECT 
    'Gold (fact_sales)' AS layer, 
    COUNT(*) AS total_records, 
    SUM(revenue) AS total_revenue
FROM ecommerce_gold.fact_sales;

-- Expected Outcome: Both rows must show the exact same count and sum.

In [0]:
-- ------------------------------------------------------------------------------
-- CHECK 2: Primary Key Uniqueness in Dimensions
-- Objective: Confirm dimensions have 100% unique primary keys (no duplicates)
-- ------------------------------------------------------------------------------

SELECT 'dim_customers' AS dimension, COUNT(*) - COUNT(DISTINCT customer_key) AS duplicate_keys FROM ecommerce_gold.dim_customers
UNION ALL
SELECT 'dim_products' AS dimension, COUNT(*) - COUNT(DISTINCT product_key) AS duplicate_keys FROM ecommerce_gold.dim_products
UNION ALL
SELECT 'dim_date' AS dimension, COUNT(*) - COUNT(DISTINCT date_key) AS duplicate_keys FROM ecommerce_gold.dim_date;

-- Expected Outcome: duplicate_keys must be 0 for all rows.

In [0]:
-- ------------------------------------------------------------------------------
-- CHECK 3: Referential Integrity (Orphan Foreign Keys in Fact Table)
-- Objective: Detect any fact row referencing non-existent dimension keys
-- ------------------------------------------------------------------------------

SELECT 
    COUNT(CASE WHEN c.customer_key IS NULL THEN 1 END) AS missing_customers,
    COUNT(CASE WHEN p.product_key IS NULL THEN 1 END)  AS missing_products,
    COUNT(CASE WHEN d.date_key IS NULL THEN 1 END)     AS missing_dates
FROM ecommerce_gold.fact_sales f
LEFT JOIN ecommerce_gold.dim_customers c ON f.customer_key = c.customer_key
LEFT JOIN ecommerce_gold.dim_products p  ON f.product_key = p.product_key
LEFT JOIN ecommerce_gold.dim_date d      ON f.date_key = d.date_key;

-- Expected Outcome: All missing_* counts must be 0.

In [0]:
-- ------------------------------------------------------------------------------
-- CHECK 4: Null Key Check in Fact Table
-- Objective: Confirm no Foreign Keys in fact_sales are NULL
-- ------------------------------------------------------------------------------

SELECT 
    COUNT(*) AS total_null_keys
FROM ecommerce_gold.fact_sales
WHERE customer_key IS NULL 
   OR product_key IS NULL 
   OR date_key IS NULL;
   
-- Expected Outcome: total_null_keys must be 0.